In [1]:
pip install transformers[torch] datasets scikit-learn

In [2]:
#mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

#read file
path = '/content/drive/My Drive/CSV/shopee_reviews_cleaned.csv'
df = pd.read_csv(path)

df.head()

,content_cleaned,label
0,aplikasinya ribet masa mengatur email error te...,0.0
1,ok,1.0
2,apk ribet gjls tll,0.0
3,sangat bermempaat,0.0
4,pengalaman ber blanja di onlaensangat membantu...,1.0


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset


# Perhatikan penggunaan 'My Drive' (pakai spasi) setelah '/content/drive/'
#path = '/content/drive/My Drive/CSV/shopee_reviews_cleaned.csv'
#df = pd.read_csv(path)

df['content_cleaned'] = df['content_cleaned'].fillna('') # Isi data kosong dengan string kosong
df['content_cleaned'] = df['content_cleaned'].astype(str) # Pastikan semua tipe data adalah string

# Ubah label ke format integer (0 dan 1)
df['label'] = df['label'].astype(int)

# 2. Split data: 80% Latihan, 20% Ujian
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# 3. Load Tokenizer (Penerjemah kata ke angka untuk IndoBERT)
model_name = "indobenchmark/indobert-base-p1"
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["content_cleaned"], padding="max_length", truncation=True, max_length=128)

# Ubah ke format Dataset Hugging Face
train_dataset = Dataset.from_dict(train_df).map(tokenize_function, batched=True)
val_dataset = Dataset.from_dict(val_df).map(tokenize_function, batched=True)

# 4. Load Model
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 5. Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",      # <--- Ganti dari evaluation_strategy menjadi eval_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
)

# 6. Mulai Training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Memulai training... ")
trainer.train()

# 7. Simpan Modelnya
model.save_pretrained("./indobert-shopee-sentiment-model")
tokenizer.save_pretrained("./indobert-shopee-sentiment-model")
print("Model berhasil disimpan di folder 'indobert-shopee-sentiment-model'!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/768 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss
1,No log,0.269567
2,No log,0.345741
3,No log,0.399630


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model berhasil disimpan di folder 'indobert-shopee-sentiment-model'!


In [5]:
import os

os.listdir('/content')

['.config',
 'drive',
 'results',
 'indobert-shopee-sentiment-model',
 'sample_data']

In [6]:
model.save_pretrained('/content/drive/MyDrive/ML_Project/indobert-shopee-sentiment-model')
tokenizer.save_pretrained('/content/drive/MyDrive/ML_Project/indobert-shopee-sentiment-model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/ML_Project/indobert-shopee-sentiment-model/tokenizer_config.json',
 '/content/drive/MyDrive/ML_Project/indobert-shopee-sentiment-model/tokenizer.json')